In [ ]:
# 코드 가동에 필요한 라이브러리 접지(import).
import os # 운영체제와 상호작용하기 위한 라이브러리 (파일/디렉토리 조작)
import time # 시간 관련 함수를 제공하는 라이브러리 (지연, 시간 측정 등)
import re # 정규표현식을 사용한 문자열 패턴 매칭 라이브러리
import json # JSON 데이터 형식의 인코딩/디코딩을 위한 라이브러리
import requests # HTTP 요청을 간편하게 보낼 수 있는 라이브러리
import pandas as pd # 데이터 분석 및 조작을 위한 강력한 데이터프레임 라이브러리
from tqdm import tqdm # 반복문의 진행 상황을 시각적으로 표시하는 진행바 라이브러리
from bs4 import BeautifulSoup # HTML/XML 문서를 파싱하고 조작하기 위한 라이브러리
from selenium import webdriver # 웹 브라우저를 자동화하여 동적 웹페이지를 제어하는 라이브러리
from selenium.webdriver.chrome.options import Options  # Chrome 브라우저의 실행 옵션을 설정하기 위한 클래스
import chromedriver_autoinstaller  # ChromeDriver를 자동으로 설치하고 관리하는 라이브러리
import ssl  # SSL 인증서 처리를 위한 라이브러리
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse  # URL 파싱 및 조작 라이브러리

In [ ]:
# 크롬 드라이버 설정
def setup_chromedriver():
    import ssl
    import certifi

    # SSL 인증서 문제 해결(mac에서 발생하는 에러 해결)
    ssl._create_default_https_context = ssl._create_unverified_context

    chromedriver_autoinstaller.install()  # Automatically installs the compatible version

    options = Options()
    options.add_argument('--headless')  # GUI 없는 환경에서 실행
    options.add_argument('--no-sandbox')  # 권한 문제 방지 (Linux)
    options.add_argument('--disable-dev-shm-usage')  # 메모리 사용 제한 해제 (Linux)
    options.add_argument('lang=ko_KR')  # 한국어 설정
    return webdriver.Chrome(options=options)

# 디렉토리 생성 함수
def create_output_dir(base_dir, sub_dir):
    output_dir = os.path.join(base_dir, sub_dir)
    os.makedirs(output_dir, exist_ok=True)
    return output_dir

# 이미지 화질 조정 함수
def adjust_image_quality(url, quality='medium'):
    """
    네이버 이미지 URL의 화질을 조정하는 함수

    Args:
        url: 원본 이미지 URL
        quality: 'high' (원본), 'medium' (w860), 'low' (w647)

    Returns:
        str: 화질이 조정된 이미지 URL
    """
    if not url or 'imgnews.pstatic.net' not in url:
        return url

    try:
        parsed = urlparse(url)
        query_params = parse_qs(parsed.query)

        # 화질 설정
        if quality == 'high':
            # 원본 화질 - type 파라미터 제거
            query_params.pop('type', None)
        elif quality == 'medium':
            query_params['type'] = ['w860']
        elif quality == 'low':
            query_params['type'] = ['w647']

        # URL 재구성
        new_query = urlencode(query_params, doseq=True)
        new_parsed = parsed._replace(query=new_query)
        return urlunparse(new_parsed)

    except Exception:
        return url

# 네이버 뉴스 링크 수집 함수 (날짜 제한 개선)
def collect_naver_news_links(query, news_office_id, start_date, end_date, max_pages=10000):
    base_url = "https://search.naver.com/search.naver?where=news"
    driver = setup_chromedriver()
    links = []

    try:
        # 날짜 형식 변환 (YYYYMMDD → YYYY.MM.DD)
        display_start_date = f"{start_date[:4]}.{start_date[4:6]}.{start_date[6:]}"
        display_end_date = f"{end_date[:4]}.{end_date[4:6]}.{end_date[6:]}"

        # 진행 상황을 표시할 tqdm 설정 (한 줄로 유지)
        with tqdm(total=max_pages, desc="링크 수집", bar_format='{l_bar}{bar:30}{r_bar}',
                 ncols=80, position=0, leave=True) as pbar:
            for page in range(1, max_pages + 1):
                # 네이버 검색 URL 형식 준수 (pd=3 추가, ds/de 추가)
                url = (f"{base_url}&query={query}&start={(page - 1) * 10 + 1}"
                       f"&pd=3&ds={display_start_date}&de={display_end_date}"
                       f"&mynews=1&office_type=1&office_section_code=1"
                       f"&news_office_checked={news_office_id}"
                       f"&nso=so:r,p:from{start_date}to{end_date},a:all")

                driver.get(url)
                time.sleep(2)  # 페이지 로딩 대기, 페이지 로딩 대기가 없을 경우, 서버에 과도한 부담을 주어 IP가 차단될 수 있음.

                # BeautifulSoup으로 파싱
                soup = BeautifulSoup(driver.page_source, 'html.parser')

                # 결과가 없는지 확인
                no_result = soup.select_one("div.api_noresult_wrap")
                if no_result:
                    print(f"\n더 이상 결과가 없습니다. (페이지 {page}에서 중단)")
                    pbar.update(max_pages - page + 1)
                    break

                # 네이버 뉴스 링크 찾기
                naver_news_elements = []

                # 1. 뉴스 카드에서 네이버뉴스 텍스트가 있는 항목 찾기
                news_cards = soup.select("div.news_area")
                for card in news_cards:
                    # 네이버뉴스 표시가 있는지 확인
                    info_spans = card.select("span")
                    is_naver_news = False
                    for span in info_spans:
                        if "네이버뉴스" in span.text:
                            is_naver_news = True
                            break

                    if is_naver_news:
                        # 해당 카드의 링크 찾기
                        links_in_card = card.select("a[href*='n.news.naver.com']")
                        naver_news_elements.extend(links_in_card)

                # 2. 일반적인 방법으로도 찾기 (백업)
                if not naver_news_elements:
                    naver_news_elements = soup.select("a[href*='n.news.naver.com']")

                # 결과가 없으면 중단
                if not naver_news_elements and page > 1:
                    print(f"\n더 이상 네이버 뉴스 링크가 없습니다. (페이지 {page}에서 중단)")
                    pbar.update(max_pages - page + 1)
                    break

                # 네이버 뉴스 링크만 수집
                for element in naver_news_elements:
                    if 'n.news.naver.com' in element['href']:
                        links.append(element['href'])

                # 중복 제거
                links = list(set(links))

                # 진행 상황 업데이트
                pbar.update(1)
                pbar.set_postfix({"링크": len(links)})

    except Exception as e:
        print(f"링크 수집 중 오류: {e}")
    finally:
        driver.quit()

    print(f"총 {len(links)}개의 뉴스 링크를 수집했습니다.")
    print(f"수집 기간: {display_start_date} ~ {display_end_date}")
    return links

# 이미지 URL 수집 함수
def extract_image_urls(soup, collect_image=True, image_quality='medium'):
    """
    BeautifulSoup 객체에서 이미지 URL들을 추출하는 함수 (Yandex 방식 적용)

    ★ 핵심: 기사 본문(article#dic_area) 내부의 이미지만 추출 ★

    Args:
        soup: BeautifulSoup 파싱 객체
        collect_image: 이미지 수집 여부
        image_quality: 'high', 'medium', 'low'
    Returns:
        list: 이미지 URL 리스트
    """
    if not collect_image:
        return []

    image_urls = []

    try:
        # ★ 핵심: 기사 본문 영역만 선택 ★
        article_content = soup.select_one("article#dic_area")

        if not article_content:
            # article#dic_area가 없으면 다른 본문 영역 시도
            article_content = soup.select_one("article")

        if not article_content:
            # 본문 영역을 찾을 수 없으면 빈 리스트 반환
            return []

        # 1단계: 기사 본문 내부의 id가 img + 숫자인 이미지들
        # img1, img2, img3 등만 선택 (img_banner, img_logo 같은 건 제외)
        all_imgs_in_article = article_content.select("img")

        for img in all_imgs_in_article:
            img_id = img.get('id', '')
            # id가 'img'로 시작하고 그 뒤가 숫자인지 확인
            if img_id and img_id.startswith('img') and len(img_id) > 3 and img_id[3:].isdigit():
                # srcset 속성 우선 확인 (Yandex 방식)
                srcset = img.get('srcset', '')
                if srcset:
                    # srcset에서 첫 번째 이미지 URL 추출
                    first_url = srcset.split(',')[0].strip().split(' ')[0]
                    if 'imgnews.pstatic.net' in first_url:
                        image_urls.append(first_url)
                else:
                    # src 속성 사용
                    src = img.get('src', '')
                    if src and 'imgnews.pstatic.net' in src:
                        image_urls.append(src)

        # 2단계: id는 없지만 _LAZY_LOADING 클래스를 가진 이미지들 (본문 내부만)
        if not image_urls:
            lazy_imgs = article_content.select("img[class*='_LAZY_LOADING']")
            for img in lazy_imgs:
                # srcset 우선 확인
                srcset = img.get('srcset', '')
                if srcset:
                    first_url = srcset.split(',')[0].strip().split(' ')[0]
                    if 'imgnews.pstatic.net' in first_url:
                        image_urls.append(first_url)
                else:
                    src = img.get('src', '') or img.get('data-src', '')
                    if src and 'imgnews.pstatic.net' in src:
                        image_urls.append(src)

        # 3단계: 본문 내부의 모든 네이버 뉴스 이미지 (최후 백업)
        if not image_urls:
            for img in all_imgs_in_article:
                srcset = img.get('srcset', '')
                if srcset:
                    first_url = srcset.split(',')[0].strip().split(' ')[0]
                    if 'imgnews.pstatic.net' in first_url:
                        image_urls.append(first_url)
                else:
                    src = img.get('src', '')
                    if src and 'imgnews.pstatic.net' in src:
                        image_urls.append(src)

        # 중복 제거 및 빈 값 필터링 (순서 유지)
        image_urls = list(dict.fromkeys(image_urls))  # 순서 유지하면서 중복 제거
        image_urls = [url for url in image_urls if url and url.strip()]

        # 화질 조정
        image_urls = [adjust_image_quality(url, image_quality) for url in image_urls]

    except Exception as e:
        print(f"이미지 URL 추출 중 오류: {e}")

    return image_urls


# ── 문자열 정리 헬퍼 (두 번째 코드의 구체화된 변인 추출에 필요) ─────────────────
def _clean_text(s):
    """공백류 문자 정리 (줄바꿈은 보존)"""
    return re.sub(r"[ \t\xa0\u200b]+", " ", str(s or "")).strip()


def _flat_text(s):
    """줄바꿈까지 모두 한 줄 공백으로 정리 (2차 첨부 코드의 flat() 과 동일)"""
    return re.sub(r"\s+", " ", str(s or "")).strip()


# ── 장르(기사 유형) 추정 — 2번째 코드(NaverNewsCommentconditioncrawler)의 변인 구체화 반영 ──
GENRE_NAME = {1: "스트레이트", 2: "해설/분석", 3: "인터뷰", 4: "사설", 5: "칼럼/기고", 6: "기타"}
COLUMN_MARKS = ["칼럼", "기고", "시론", "발언대", "특별기고", "태평로", "동서남북", "에스프레소",
                "아침햇발", "세상읽기", "유레카", "만물상", "서초포럼", "뉴스룸에서", "현장에서",
                "朝鮮칼럼", "편집국에서", "슬기로운 기자생활"]
ANALYSIS_MARKS = ["뷰리핑", "뉴스 다이브", "논썰", "뉴스 저격", "분석", "팩트체크", "해설",
                  "이슈 인사이드", "심층", "흑백여의도"]
INTERVIEW_MARKS = ["인터뷰", "일문일답", "대담"]


def detect_genre(title, section, author):
    br = " ".join(re.findall(r"[\[\(【]([^\]\)】]{1,16})[\]\)】]", title or ""))
    title = title or ""
    section = section or ""
    author = author or ""
    if "사설" in br:
        return 4, "제목[사설]"
    if any(k in br for k in COLUMN_MARKS) or "칼럼" in title:
        return 5, "제목 칼럼마커"
    if any(k in author for k in ["논설위원", "논설실장", "주필", "편집인", "에디터"]):
        return 5, "바이라인 논설"
    if author and "기자" not in author and "특파원" not in author and len(author) > 2:
        return 5, "바이라인 외부필자"
    if any(k in br for k in INTERVIEW_MARKS) or "인터뷰" in title:
        return 3, "제목 인터뷰"
    if section == "오피니언":
        return 5, "섹션 오피니언"
    if any(k in br for k in ANALYSIS_MARKS):
        return 2, "제목 분석마커"
    return 1, "기본값"


# 기사 데이터 수집 함수
# ── 기존 크롤러의 기사 수집(요청·파싱) 로직은 그대로 두고, 수집 변인만
#    두 번째 코드(NaverNewsCommentconditioncrawler)의 구체화된 정의를 반영해 확장한다. ──
def collect_article_details(media_name, url, collect_image=True, image_quality='medium'):
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.content, "html.parser")

        # 제목 추출 (기존 방식 유지)
        title = ""

        # 1. 가장 정확한 제목 선택자로 시도
        title_span = soup.select_one("h2#title_area > span")
        if title_span:
            title = title_span.text.strip()

        # 2. 클래스로 찾기
        if not title:
            title_span = soup.select_one("h2.media_end_head_headline > span")
            if title_span:
                title = title_span.text.strip()

        # 3. 제목 영역 div에서 찾기
        if not title:
            title_div = soup.select_one("div.media_end_head_title")
            if title_div:
                spans = title_div.select("span")
                for span in spans:
                    span_text = span.text.strip()
                    if len(span_text) > 10:  # 제목으로 의미있는 길이
                        title = span_text
                        break

        # 4. 이전 방식으로 시도 (폴백)
        if not title:
            for h_tag in ['h1', 'h2', 'h3']:
                headers = soup.select(h_tag)
                for header in headers:
                    header_text = header.text.strip()
                    if header_text and len(header_text) > 10:
                        title = header_text
                        break
                if title:
                    break

        # 5. 마지막 방법: 긴 span 태그 찾기 (하지만 "구독해주세요"와 같은 불필요한 텍스트는 필터링)
        if not title:
            avoid_words = ["구독", "좋아요", "팔로우", "뉴스레터", "공유", "신청", "클릭"]
            for span in soup.select("span"):
                span_text = span.text.strip()
                # 광고나 구독 문구가 아닌 실제 제목만 추출
                if (span_text and len(span_text) > 15 and
                    not any(word in span_text for word in avoid_words)):
                    title = span_text
                    break

        # 작성자 추출
        writer = ""
        writer_element = soup.select_one("span.byline_s")
        if writer_element:
            writer = writer_element.text.strip()

        # ── 날짜/시각 추출 (구체화) ──────────────────────────────────────────
        # 2번째 코드처럼 표시 문자열이 아니라 data-date-time 속성(원자료)을 기준으로 삼는다.
        date = ""
        time_only = ""
        datetime_iso = ""
        date_el = soup.select_one("span._ARTICLE_DATE_TIME")
        raw_date = date_el.get("data-date-time", "").strip() if date_el else ""
        if raw_date:
            date = raw_date[:10]
            time_only = raw_date[11:16] if len(raw_date) >= 16 else ""
            if len(raw_date) >= 19:
                datetime_iso = f"{raw_date[:10]}T{raw_date[11:19]}+0900"
        else:
            # 폴백: 기존(첫 번째 코드) 방식 — 화면 표시 문자열 탐색
            date_elements = soup.select("span[class*='ARTICLE_DATE_TIME']")
            if date_elements:
                date = date_elements[0].text.strip()
            if not date:
                for span in soup.select("span"):
                    span_text = span.text.strip()
                    if ("202" in span_text) and ("." in span_text or ":" in span_text):
                        if re.search(r'\d{4}[./-]\d{1,2}[./-]\d{1,2}', span_text):
                            date = span_text
                            break

        # 수정 시각 (원본 속성값 + ISO 결합본 둘 다 보존)
        modify_date = ""
        modify_datetime = ""
        modify_el = soup.select_one("span._ARTICLE_MODIFY_DATE_TIME")
        if modify_el:
            raw_modify = modify_el.get("data-modify-date-time", "").strip()
            modify_date = raw_modify
            if raw_modify and len(raw_modify) >= 19:
                modify_datetime = f"{raw_modify[:10]}T{raw_modify[11:19]}+0900"

        # 섹션 정보 추출
        section = ""
        # 1. 'media_end_categorize_item' 클래스로 시도
        section_element = soup.select_one("em.media_end_categorize_item")
        if section_element:
            section = section_element.text.strip()

        # 2. 다른 선택자 시도 (백업)
        if not section:
            # 다른 가능한 섹션 선택자들 시도
            section_selectors = [
                "div.media_end_categorize_item",
                "a.categorize_item",
                "div.article_category",
                "span.category"
            ]

            for selector in section_selectors:
                section_elem = soup.select_one(selector)
                if section_elem and section_elem.text.strip():
                    section = section_elem.text.strip()
                    break

        # 3. 마지막 백업: 메타 태그에서 찾기
        if not section:
            meta_keywords = soup.select_one("meta[property='article:section'], meta[name='article:section']")
            if meta_keywords and meta_keywords.get('content'):
                section = meta_keywords.get('content')

        # 이미지 URL 추출 (기존 방식 유지)
        image_urls = extract_image_urls(soup, collect_image, image_quality)

        # ── 신규 수집 변수 (기존 크롤러에 이미 있던 것들) ───────────────────

        # PICK 여부 (언론사 자체 강조 기사)
        pick = 1 if soup.select_one("i.media_end_head_channel_pick") else 0

        # 면 정보 (예: '4면 1단', 'A1면 3단' — 언론사마다 형식 상이 → raw 저장)
        page = ""
        for span in soup.select("span.sds-comps-text-weight-sm"):
            span_text = span.text.strip()
            if "면" in span_text:
                page = span_text
                break

        # 전체 섹션 목록 (다중 섹션 분류 지원)
        section_items = [el.text.strip() for el in soup.select("em.media_end_categorize_item") if el.text.strip()]
        sections = "|".join(section_items)
        sec_count = len(section_items)

        # 반응("이 기사를 추천합니다") 수집 — 총합 + 5종 세분화
        #   쏠쏠정보(useful) · 흥미진진(wow) · 공감백배(touched) · 분석탁월(analytical) · 후속강추(recommend)
        # ★ ul._faceLayer / li.u_likeit_list 같은 감싸는 클래스에 의존하지 않고,
        #   data-type 속성을 가진 <a> 태그를 문서 전체에서 직접 찾는다.
        #   (감싸는 wrapper의 클래스명은 페이지 버전에 따라 바뀔 수 있지만,
        #    data-type="useful/wow/touched/analytical/recommend" 값 자체는 안정적이다.)
        react_total = 0
        react_useful = react_wow = react_touched = react_analytic = react_recommend = 0
        react_total_el = soup.select_one("span.u_likeit_text._count")
        if react_total_el:
            try: react_total = int(react_total_el.text.strip().replace(",", ""))
            except: pass

        REACT_TYPE_MAP = {
            "useful": "useful", "wow": "wow", "touched": "touched",
            "analytical": "analytical", "recommend": "recommend",
        }
        for btn in soup.select("a[data-type]"):
            dtype = btn.get("data-type", "")
            if dtype not in REACT_TYPE_MAP:
                continue  # 감정표현(good/warm/sad/angry/want) 등 다른 위젯의 data-type은 제외
            cnt = btn.select_one("span._count") or btn.select_one("span.u_likeit_list_count")
            if not cnt:
                continue
            try: val = int(cnt.text.strip().replace(",", ""))
            except: val = 0
            if dtype == "useful":       react_useful    = val
            elif dtype == "wow":        react_wow       = val
            elif dtype == "touched":    react_touched   = val
            elif dtype == "analytical": react_analytic  = val
            elif dtype == "recommend":  react_recommend = val

        # AI 요약 제공 여부
        ai_summary = 1 if soup.select_one("div#_SUMMARY_BUTTON") else 0

        # 본문 내 동영상 포함 여부
        article_body = soup.select_one("article#dic_area")
        has_video = 0
        if article_body:
            has_video = 1 if article_body.select_one("div._VIDEO_AREA, video, iframe[src*='video']") else 0

        # 언론사 주요뉴스 아웃링크 수 (자기참조 밀도)
        outlink_n = len(soup.select("ul.media_end_linked_list > li"))

        # 연결 이슈 수 및 이슈명 (의제 프레이밍 네트워크)
        issue_items = [el.text.strip() for el in soup.select("strong.related_issue_subject_name") if el.text.strip()]
        issue_n = len(issue_items)
        issues  = "|".join(issue_items)

        # 기자 네이버 ID
        journalist_id = ""
        jid_el = soup.select_one("div._JOURNALIST_CARD div._JOURNALIST_ID")
        if jid_el:
            journalist_id = jid_el.get("data-journalistid", "").strip()

        # 원문 링크 존재 여부 (플랫폼 종속성)
        has_origin = 1 if soup.select_one("a.media_end_head_origin_link") else 0

        # TTS 서비스 제공 여부 (포맷 다양화)
        has_tts = 1 if soup.select_one("div.media_end_head_tts") else 0

        # 기사 고유 ID (URL에서 파싱 — 패널·종단 분석용 키)
        article_id = ""
        import re as _re
        aid_match = _re.search(r'/article/\d+/(\d+)', url)
        if aid_match:
            article_id = aid_match.group(1)

        # 언론사 oid / 기사 aid 분리 저장 (2번째 코드의 naver_oid/naver_aid 구체화 반영)
        naver_oid, naver_aid = "", ""
        oid_aid_match = _re.search(r"/article/(?:comment/)?(\d{3,4})/(\d{7,11})", url)
        if oid_aid_match:
            naver_oid, naver_aid = oid_aid_match.group(1), oid_aid_match.group(2)

        # 언론사 유형 (JS 변수에서 파싱: '방송/통신사', '종합일간지' 등)
        office_type = ""
        raw_html = response.text
        otype_match = _re.search(r'office\s*=\s*\{[^}]*category\s*:\s*"([^"]+)"', raw_html)
        if otype_match:
            office_type = otype_match.group(1).strip()

        # og:description (네이버 생성 요약 — 본문과 의미 괴리 분석용)
        og_desc = ""
        og_el = soup.select_one("meta[property='og:description']")
        if og_el:
            og_desc = og_el.get("content", "").strip()

        # 댓글 허용 여부 (cbox JS 내 차단 문구 탐지)
        comment_allow = 0 if 'info_comment_off' in raw_html else 1

        # 네이버 섹션 코드 (URL sid 파라미터 — 언론사 분류와 교차 검증용)
        naver_sid = ""
        sid_match = _re.search(r'[?&]sid=(\d+)', url)
        if sid_match:
            naver_sid = sid_match.group(1)

        # 기사 포맷 유형 (JS article.type: 0=텍스트, 1=포토, 2=TV)
        article_type = ""
        atype_match = _re.search(r'article\s*=\s*\{[^}]*type:\s*"(\d)"', raw_html)
        if atype_match:
            article_type = atype_match.group(1)

        # 대표 썸네일 URL (og:image)
        thumb_url = ""
        og_img_el = soup.select_one("meta[property='og:image']")
        if og_img_el:
            thumb_url = og_img_el.get("content", "").strip()

        # 댓글 개수 (id="comment_count" / class="_COMMENT_COUNT_VIEW")
        # ★ 실측 결과, 정적 HTML(requests) 안의 이 span 은 항상 "댓글"이라는
        #   자리표시자 문자열만 담고 있고 실제 숫자는 JS 로 채워진다(브라우저에서만 보임).
        #   그래서 span 텍스트를 그대로 파싱하면 항상 0으로 남는다.
        #   실제 숫자는 댓글 위젯이 내부적으로 호출하는 cbox 카운트 API 응답에만 있어서,
        #   여기서 같은 API를 가벼운 1페이지 요청으로 한 번 더 불러 그 값을 가져온다.
        comment_n = 0
        if naver_oid and naver_aid:
            try:
                _cc_headers = {
                    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                                  "(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
                    "Accept": "application/json, text/javascript, */*; q=0.01",
                    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8",
                    "X-Requested-With": "XMLHttpRequest",
                    "Referer": url,
                }
                _cc_params = {
                    "ticket": "news", "templateId": "default_society", "pool": "cbox5",
                    "lang": "ko", "country": "KR",
                    "objectId": f"news{naver_oid},{naver_aid}",
                    "pageSize": 1, "indexSize": 1, "page": 1, "sort": "favorite",
                    "includeAllStatus": "true",
                }
                _cc_resp = requests.get("https://apis.naver.com/commentBox/cbox/web_naver_list_jsonp.json",
                                         params=_cc_params, headers=_cc_headers, timeout=10)
                _cc_m = _re.search(r"\{.*\}", _cc_resp.text, _re.DOTALL)
                if _cc_m:
                    _cc_data = json.loads(_cc_m.group(0))
                    comment_n = int(_cc_data.get("result", {}).get("count", {}).get("comment", 0) or 0)
            except Exception:
                pass

        # 네이버 전역 기사 ID (gdid — 고유 식별 보완 키)
        gdid = ""
        gdid_match = _re.search(r'gdid["\s]*:["\s]*"([^"]+)"', raw_html)
        if not gdid_match:
            gdid_meta = soup.select_one("meta[name='napp-site-analysis']")
            if gdid_meta:
                g2 = _re.search(r'gdid=([^,]+)', gdid_meta.get("content", ""))
                if g2:
                    gdid = g2.group(1).strip()
        else:
            gdid = gdid_match.group(1).strip()

        # ── 본문 분해: 부제·사진설명·중간제목을 본문에서 분리 ──────────────────
        # ★ 2번째 코드(NaverNewsCommentconditioncrawler)의 핵심 구체화 지점.
        #   네이버는 부제(strong.media_end_summary)와 사진설명(span.end_photo_org)을
        #   article#dic_area 안에 그대로 넣어 두므로, 분리하지 않으면 본문(text)이 오염된다.
        subtitle = captions = subheads = fulltext = ""
        img_n = 0
        art = soup.select_one("article#dic_area")
        if art:
            # 원본 soup(이미지 추출 등에 재사용됨)을 건드리지 않도록 복제본에서 작업
            a = BeautifulSoup(str(art), "html.parser").select_one("article")
            img_n = len(a.select("img"))

            subs = []
            # 1. 클래스가 명시된 공식 부제 추출
            for e in a.select("strong.media_end_summary, div.media_end_summary, h3.media_end_summary"):
                subs.append(_clean_text(e.get_text(" ", strip=True))); e.decompose()

            # 2. class가 없는 가짜 부제(본문 최상단의 b, strong 태그) 추출 및 제거
            for e in a.find_all(['b', 'strong']):
                txt = _clean_text(e.get_text(" ", strip=True))
                if not txt or len(txt) < 5:
                    continue  # 너무 짧은 단어는 무시

                # (A) 본문 시작 부근인지 확인 (이 태그 앞의 텍스트가 40자 이내인지 계산)
                prev_len = 0
                for el in a.descendants:
                    if el == e:
                        break
                    if isinstance(el, str):
                        prev_len += len(el.strip())

                if prev_len > 40:
                    continue  # 기사 중간에 등장하는 강조 텍스트면 패스

                # (B) 인라인 여부 확인 (본문의 일부인지, 독립된 부제인지)
                is_inline = False
                for el in e.next_elements:
                    if el in e.descendants:
                        continue  # 자기 자신 내부의 태그는 건너뜀

                    if isinstance(el, str):
                        if el.strip():
                            is_inline = True  # 바로 텍스트가 이어지면 본문의 일부로 간주
                            break
                    elif getattr(el, 'name', None) in ['br', 'p', 'div', 'article']:
                        break  # 텍스트가 나오기 전에 줄바꿈(<br>)을 만나면 독립된 부제가 맞음

                # 조건에 부합하면 부제로 편입하고 본문에서 삭제
                if not is_inline:
                    subs.append(txt)
                    e.decompose()

            subtitle = " | ".join(x for x in subs if x)

            caps = []
            for e in a.select("span.end_photo_org, em.img_desc, div.nbd_table"):
                c = _clean_text(e.get_text(" ", strip=True))
                if c: caps.append(c)
                e.decompose()
            captions = " || ".join(dict.fromkeys(caps))

            heads = [_clean_text(e.get_text(" ", strip=True)) for e in a.select("strong, b, h3, h4")]
            subheads = " || ".join(dict.fromkeys(h for h in heads if h and len(h) < 120))

            for e in a.select("script, style"):
                e.decompose()
            for br in a.select("br"):
                br.replace_with("\n")
            raw_txt = re.sub(r"[ \t\xa0\u200b]+", " ", a.get_text(""))
            raw_txt = "\n".join(l.strip() for l in raw_txt.split("\n"))
            fulltext = re.sub(r"\n{3,}", "\n\n", raw_txt).strip()

        # text / text_len 은 부제·사진설명이 걷힌 fulltext 기준으로 재계산한다
        # (기존 코드의 text는 본문 전체를 그대로 이어붙여 부제·사진설명이 섞여 있었다)
        text = _flat_text(fulltext)
        text_len = len(fulltext)
        fulltext_available = 1 if text_len >= 50 else 0

        # 장르(기사 유형) 추정
        genre, genre_rule = detect_genre(title, section, writer)
        genre_label = GENRE_NAME.get(genre, "")

        return {
            "media":            media_name,
            "office_type":      office_type,
            "article_id":       article_id,
            "gdid":             gdid,
            "title":            title,
            "subtitle":         subtitle,
            "url":              url,
            "naver_sid":        naver_sid,
            "naver_oid":        naver_oid,
            "naver_aid":        naver_aid,
            "date":             date,
            "time":             time_only,
            "datetime":         datetime_iso,
            "modify_date":      modify_date,
            "modify_datetime":  modify_datetime,
            "article_type":     article_type,
            "genre":            genre,
            "genre_label":      genre_label,
            "genre_rule":       genre_rule,
            "fulltext":         fulltext,
            "text":             text,
            "text_len":         text_len,
            "fulltext_available": fulltext_available,
            "captions":         captions,
            "subheads":         subheads,
            "og_desc":          og_desc,
            "thumb_url":        thumb_url,
            "writer":           writer,
            "journalist_id":    journalist_id,
            "section":          section,
            "sections":         sections,
            "sec_count":        sec_count,
            "page":             page,
            "pick":             pick,
            "react_total":      react_total,
            "react_useful":     react_useful,
            "react_wow":        react_wow,
            "react_touched":    react_touched,
            "react_analytic":   react_analytic,
            "react_recommend":  react_recommend,
            "ai_summary":       ai_summary,
            "has_video":        has_video,
            "has_tts":          has_tts,
            "has_origin":       has_origin,
            "img_n":            img_n,
            "comment_n":        comment_n,
            "comment_allow":    comment_allow,
            "outlink_n":        outlink_n,
            "issue_n":          issue_n,
            "issues":           issues,
            "images":           "|".join(image_urls)
        }
    except Exception as e:
        # 오류가 발생해도 조용히 빈 데이터 반환
        return {
            "media":            media_name,
            "office_type":      "",
            "article_id":       "",
            "gdid":             "",
            "title":            "",
            "subtitle":         "",
            "url":              url,
            "naver_sid":        "",
            "naver_oid":        "",
            "naver_aid":        "",
            "date":             "",
            "time":             "",
            "datetime":         "",
            "modify_date":      "",
            "modify_datetime":  "",
            "article_type":     "",
            "genre":            "",
            "genre_label":      "",
            "genre_rule":       "",
            "fulltext":         "",
            "text":             "",
            "text_len":         0,
            "fulltext_available": 0,
            "captions":         "",
            "subheads":         "",
            "og_desc":          "",
            "thumb_url":        "",
            "writer":           "",
            "journalist_id":    "",
            "section":          "",
            "sections":         "",
            "sec_count":        0,
            "page":             "",
            "pick":             0,
            "react_total":      0,
            "react_useful":     0,
            "react_wow":        0,
            "react_touched":    0,
            "react_analytic":   0,
            "react_recommend":  0,
            "ai_summary":       0,
            "has_video":        0,
            "has_tts":          0,
            "has_origin":       0,
            "img_n":            0,
            "comment_n":        0,
            "comment_allow":    0,
            "outlink_n":        0,
            "issue_n":          0,
            "issues":           "",
            "images":           ""
        }

# ── 병렬 댓글 수집 ───────────────────────────────────────────────────────────
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed as _as_completed

_thread_local = threading.local()

def _get_thread_session(referer_url):
    if not hasattr(_thread_local, "session"):
        s = requests.Session()
        s.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
            "Accept": "application/json, text/javascript, */*; q=0.01",
            "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8",
            "X-Requested-With": "XMLHttpRequest",
        })
        _thread_local.session = s
    _thread_local.session.headers["Referer"] = referer_url
    return _thread_local.session


# ── 댓글 전수 수집 로직 (2번째 코드의 fetch_comments 를 반영) ─────────────────
# 기존(1번 코드) 로직과 달리:
#   1) includeAllStatus=true 를 붙여 삭제·클린봇 처리된 댓글까지 모두 받는다
#      (이게 "전수(全數)" 수집의 핵심 — 화면에서 사라진 댓글도 카운트에서 빠지지 않는다)
#   2) page 파라미터만 증가시키는 것이 아니라 cbox 응답의 morePage.next 값을
#      moreParam.next 로 되돌려 보내는 방식으로 페이지네이션한다.
#      (cbox 는 page 파라미터만으로는 종종 페이지를 건너뛰거나 중복시킨다)
#   3) 신규 댓글이 0건인 페이지가 나와도 즉시 멈추지 않고, 연속 stall 허용치
#      (COMMENT_STALL_MAX)까지는 계속 다음 페이지를 시도한다.
CBOX_URL = "https://apis.naver.com/commentBox/cbox/web_naver_list_jsonp.json"
COMMENT_SORT = "old"        # old(등록순) / new / favorite
COMMENT_PAGE_MAX = 1000
COMMENT_STALL_MAX = 12      # 신규 0인 페이지를 몇 번까지 참을지 (cbox 페이지 불규칙성 대응)
COMMENT_GAP = 0.15

# 댓글에서 수집할 필드 (전수 수집이므로 삭제·클린봇 판정에 필요한 필드까지 포함)
# ★ mark: userIdNo 가 최근 네이버 쪽에서 항상 빈 값("")으로 내려오게 되면서
#   (실측 확인 — 프라이버시 정책 변경으로 추정) 대신 쓸 수 있는 필드.
#   댓글 위젯의 "댓글모음"(이 사용자의 다른 댓글 보기) 버튼이 내부적으로 쓰는
#   data-param 의 targetMark 값과 100% 동일함을 실제 기사로 검증했다 —
#   즉 네이버가 동일인 여부 조회에 실제로 쓰는 값이므로 userIdNo 의 사실상
#   대체재로 볼 수 있다. userName/maskedUserId 는 앞 4글자만 보이는 마스킹이라
#   서로 다른 사람이 같은 값으로 겹칠 수 있으니(실측으로 확인됨), 동일인 판별은
#   userName 이 아니라 이 mark 값 기준으로 하는 것이 더 정확하다.
COMMENT_FIELDS = ["commentNo", "userIdNo", "mark", "userName", "contents", "regTime", "modTime",
                   "sympathyCount", "antipathyCount", "deleted", "hiddenByCleanbot",
                   "status", "inspectionId"]

# ── 댓글 조치유형 분류 · 생존시간 분석 (2번째 코드의 judge_action 을 반영) ──────
# JSON 상태값(deleted/status/hiddenByCleanbot/inspectionId)만으로 판정한다.
# (2번째 코드의 ACTION_HTML 단계 — Selenium 으로 화면 표시 문구를 대조하는 부분은
#  전수 수집 로직과 무관한 별도 검증 기능이라 이번 반영 범위에서는 제외했다.)
ACTION_TEXT = {
    "작성자에 의해 삭제된 댓글입니다.":                             "작성자삭제",
    "운영규정 미준수로 인해 삭제된 댓글입니다.":                     "운영규정미준수삭제",
    "정보통신망법에 따른 권리침해 요청이 있어, 게시중단 되었습니다.": "권리침해게시중단",
    "클린봇이 부적절한 표현을 감지한 댓글입니다.":                   "클린봇",
}
TEXT_OF = {v: k for k, v in ACTION_TEXT.items()}
STATUS_MAP = {"1": "작성자삭제", "3": "운영규정미준수삭제"}

# get_comments() 가 COMMENT_FIELDS 뒤에 이어붙이는 분석 파생 필드
COMMENT_DERIVED_FIELDS = ["action_type", "action_text", "action_source",
                           "action_time", "survival_min", "survival_day",
                           "comment_text", "cleanbot_original",
                           "is_deleted", "is_cleanbot"]


def judge_action(c):
    """조치유형 판정 (JSON 상태값 기준). HTML 대체문구 대조 없이 status/deleted/
    hiddenByCleanbot/inspectionId 조합만으로 분류한다.

    ★ inspectionId 를 status 판정보다 먼저 확인한다 (실제 데이터로 검증된 우선순위).
      status="3"+inspectionId 있음 조합을 실제 화면 문구와 대조해보면
      "정보통신망법에 따른 권리침해 요청이 있어, 게시중단 되었습니다." 로 나타난다.
      즉 inspectionId 는 네이버가 (신고에 따른) 심사 절차를 거쳤다는 표식이므로,
      순수 커뮤니티 운영정책 위반과는 다른 사유다. status 코드만으로는 이 둘을
      구분할 수 없고, inspectionId 유무가 실질적인 구분 신호였다."""
    if c.get("hiddenByCleanbot"):
        return "클린봇", "JSON_cleanbot"
    if not c.get("deleted"):
        return "노출", "JSON_정상"
    if c.get("inspectionId"):
        return "권리침해게시중단", "JSON_inspectionId"
    st = str(c.get("status"))
    if st in STATUS_MAP:
        return STATUS_MAP[st], f"JSON_status{st}"
    return "미분류", f"status{st}_미관측"


def _augment_comment(c):
    """원본 댓글 dict(c) → COMMENT_DERIVED_FIELDS 순서의 분석값 리스트."""
    atype, src = judge_action(c)
    reg, mod = c.get("regTime", ""), c.get("modTime", "")
    acted = bool(c.get("deleted"))
    action_time = mod if acted else ""
    survival_min = survival_day = ""
    if acted and reg and mod:
        try:
            survival_min = round((pd.Timestamp(mod) - pd.Timestamp(reg)).total_seconds() / 60, 1)
            survival_day = round(survival_min / 1440, 2)
        except Exception:
            pass
    body = str(c.get("contents") or "")
    # 삭제된 댓글은 본문이 비어 있으므로 화면에 뜨는 대체문구를 comment_text 에 채운다
    comment_text = body if atype in ("노출", "클린봇") else TEXT_OF.get(atype, "")
    # 클린봇은 화면에서만 가려질 뿐 원문이 API 응답에는 그대로 남아 있다
    cleanbot_original = body if atype == "클린봇" else ""
    return [atype, TEXT_OF.get(atype, ""), src, action_time, survival_min, survival_day,
            comment_text, cleanbot_original,
            1 if acted else 0, 1 if c.get("hiddenByCleanbot") else 0]


def get_comments(story_url):
    comments = []
    try:
        object_id = None
        m = re.search(r"article/(\d{3,4})/(\d{7,11})", story_url)
        if m:
            object_id = f"{m.group(1)}/{m.group(2)}"
        if not object_id:
            m = re.search(r"(\d{3}/\d{9,11})", story_url)
            if m:
                object_id = m.group(1)
        if not object_id:
            parts = story_url.rstrip("/").split("/")
            if len(parts) >= 2:
                aid = parts[-1].split("?")[0]
                oid = parts[-2]
                if aid.isdigit() and oid.isdigit():
                    object_id = f"{oid}/{aid}"
        if not object_id:
            return comments

        fmt_id = object_id.replace("/", "%2C")
        session = _get_thread_session(story_url)
        seen = set()
        next_param = ""
        stall = 0

        for page in range(1, COMMENT_PAGE_MAX + 1):
            params = {
                "ticket": "news", "templateId": "default_society", "pool": "cbox5",
                "lang": "ko", "country": "KR", "objectId": f"news{fmt_id}",
                "pageSize": 100, "indexSize": 10, "page": page, "sort": COMMENT_SORT,
                "includeAllStatus": "true",
            }
            if next_param:
                params["pageType"] = "more"
                params["moreParam.next"] = next_param

            resp = session.get(CBOX_URL, params=params, timeout=10)
            mm = re.search(r"_callback\((.*)\);", resp.text)
            if mm:
                json_str = mm.group(1)
            else:
                m2 = re.search(r"\{.*\}", resp.text, re.DOTALL)
                if not m2:
                    break
                json_str = m2.group(0)

            try:
                data = json.loads(json_str)
            except Exception:
                break

            result = data.get("result", {}) or {}
            clist = result.get("commentList", [])

            new = 0
            for c in clist:
                no = str(c.get("commentNo", "")) or f"{c.get('userIdNo','')}_{c.get('regTime','')}"
                if no not in seen:
                    seen.add(no)
                    row = [c.get(f, "") for f in COMMENT_FIELDS] + _augment_comment(c)
                    comments.append(row)
                    new += 1

            # 다음 페이지 계산: morePage.next 를 우선으로 사용 (cbox 페이지네이션 불규칙성 대응)
            more = result.get("morePage", {}) or {}
            next_param = more.get("next", "")

            if not clist:
                break

            stall = stall + 1 if new == 0 else 0
            if stall >= COMMENT_STALL_MAX:
                break
            if not more or not next_param:
                # next 값이 더 없으면 더 이상 가져올 페이지가 없다는 뜻
                break

            time.sleep(COMMENT_GAP)
    except Exception as e:
        print(f"  댓글 수집 오류: {story_url}: {e}")
    return comments


def _comment_worker(args):
    row, media_name = args
    url = row.get("url", "")
    if not isinstance(url, str) or not url:
        return []
    try:
        comments = get_comments(url)
        return [
            [url, media_name, row.get("title",""), row.get("text",""), row.get("images","")] + c
            for c in comments
        ]
    except Exception as e:
        print(f"  워커 오류: {url}: {e}")
        return []


def collect_and_save_comments(base_path, output_file_prefix, max_workers=5):
    import csv as _csv
    COLS = ["url", "media", "title", "text", "images"] + COMMENT_FIELDS + COMMENT_DERIVED_FIELDS
    total = 0
    comments_dir = os.path.join(base_path, "comments")
    os.makedirs(comments_dir, exist_ok=True)

    for media_name in os.listdir(base_path):
        media_dir = os.path.join(base_path, media_name)
        if not os.path.isdir(media_dir) or media_name == "comments":
            continue
        article_file = os.path.join(media_dir, "articles.csv")
        if not os.path.isfile(article_file):
            continue

        print(f"  [{media_name}] 댓글 전수 수집 (workers={max_workers})")
        try:
            df = pd.read_csv(article_file)
        except Exception as e:
            print(f"  로드 오류: {e}"); continue

        rows = [r.to_dict() for _, r in df.iterrows()]
        buf = []

        with tqdm(total=len(rows), desc=f"{media_name}",
                  bar_format="{l_bar}{bar:30}{r_bar}", ncols=80) as pbar:
            with ThreadPoolExecutor(max_workers=max_workers) as ex:
                futs = {ex.submit(_comment_worker, (r, media_name)): r for r in rows}
                for fut in _as_completed(futs):
                    buf.extend(fut.result())
                    pbar.update(1)
                    if len(buf) >= 10000:
                        pd.DataFrame(buf, columns=COLS).to_csv(
                            os.path.join(comments_dir,
                                f"{output_file_prefix}_{media_name}_part{total}.csv"),
                            index=False, encoding="utf-8-sig", quoting=_csv.QUOTE_ALL)
                        total += len(buf); buf = []

        if buf:
            pd.DataFrame(buf, columns=COLS).to_csv(
                os.path.join(comments_dir, f"{output_file_prefix}_{media_name}.csv"),
                index=False, encoding="utf-8-sig", quoting=_csv.QUOTE_ALL)
            total += len(buf)
            print(f"  ✓ {media_name}: {len(buf)}개 저장")

    with open(os.path.join(comments_dir, f"{output_file_prefix}_summary.txt"),
              "w", encoding="utf-8") as f:
        f.write(f"총 댓글: {total}\n완료: {time.strftime('%Y-%m-%d %H:%M:%S')}\nworkers: {max_workers}\n")
    print(f"\n★ 총 {total}개 댓글 수집 완료 → {comments_dir}")

    # ── 조치유형 분류 · 생존시간 분석 (방금 저장한 댓글 CSV 전체를 다시 읽어 집계) ──
    analyze_comment_actions(comments_dir, output_file_prefix)


def analyze_comment_actions(comments_dir, output_file_prefix):
    """comments_dir 에 저장된 댓글 CSV 를 모두 모아 조치유형·생존시간을 집계하고
    콘솔에 출력 + {prefix}_action_summary.txt 로 저장한다."""
    import glob
    csv_files = [p for p in glob.glob(os.path.join(comments_dir, f"{output_file_prefix}_*.csv"))
                 if not p.endswith("_summary.txt")]
    if not csv_files:
        return

    dfs = []
    for p in csv_files:
        try:
            dfs.append(pd.read_csv(p))
        except Exception as e:
            print(f"  [분석] 로드 오류: {p}: {e}")
    if not dfs:
        return
    cd = pd.concat(dfs, ignore_index=True)
    if "action_type" not in cd.columns or len(cd) == 0:
        return

    lines = []
    lines.append(f"댓글 총 {len(cd):,}건")

    lines.append("\n[조치유형]")
    for k, v in cd["action_type"].value_counts().items():
        lines.append(f"  {k:20s} {v:6,}")

    lines.append("\n[판정근거]")
    for k, v in cd["action_source"].value_counts().items():
        lines.append(f"  {k:20s} {v:6,}")

    cb = cd[cd["action_type"] == "클린봇"]
    if len(cb):
        n = (cb["cleanbot_original"].astype(str).str.strip() != "").sum()
        lines.append(f"\n[클린봇] {len(cb):,}건 → 가려진 원문 복원 {n:,}건 "
                      f"({n/len(cb)*100:.1f}%)")

    sv = pd.to_numeric(cd["survival_min"], errors="coerce").dropna()
    if len(sv):
        lines.append(f"\n[생존시간] 작성 → 조치까지  n={len(sv):,}")
        lines.append(f"  중앙 {sv.median():,.0f}분 ({sv.median()/1440:.1f}일) · "
                      f"평균 {sv.mean():,.0f}분")
        lines.append(f"  1시간내 {(sv<60).mean()*100:.1f}% · 1일내 {(sv<1440).mean()*100:.1f}% · "
                      f"7일내 {(sv<10080).mean()*100:.1f}%")
        g = (cd.assign(_s=pd.to_numeric(cd["survival_min"], errors="coerce"))
               .dropna(subset=["_s"]).groupby("action_type")["_s"]
               .agg(["count", "median", "mean"]).round(1))
        lines.append("\n[조치유형별 생존시간(분)]")
        lines.append(g.to_string())

    report = "\n".join(lines)
    print("\n" + report)
    with open(os.path.join(comments_dir, f"{output_file_prefix}_action_summary.txt"),
              "w", encoding="utf-8") as f:
        f.write(report + "\n")
    print(f"\n★ 조치유형·생존시간 분석 저장 → "
          f"{os.path.join(comments_dir, output_file_prefix + '_action_summary.txt')}")

# ── 언론사 전체 기사 링크 수집 (키워드 없이, 날짜 루프) ── ※ 기존 로직 그대로 유지 ──
def collect_office_news_links(news_office_id, start_date, end_date, max_pages=10000):
    """
    특정 언론사(oid)의 기간 내 '전체 기사' 링크를 수집한다 (검색어 미사용).
    use_query=False 일 때 collect_naver_news_links 대신 호출된다.

    ★★ 주의 ★★ 아래 목록 엔드포인트(list.naver)는 네이버 구조 변경이 잦다.
    실행 전 반드시 한 언론사·하루로 링크가 실제 잡히는지 검증하고,
    막혀 있으면 'https://media.naver.com/press/{oid}' 피드/지면 엔드포인트로 교체할 것.
    """
    from datetime import datetime, timedelta
    oid = f"{news_office_id - 1000:03d}"   # 1023 → '023', 1001 → '001'
    driver = setup_chromedriver()
    links = []
    try:
        d0 = datetime.strptime(start_date, "%Y%m%d")
        d1 = datetime.strptime(end_date,   "%Y%m%d")
        total_days = (d1 - d0).days + 1
        with tqdm(total=total_days, desc=f"[oid {oid}] 전체기사 링크",
                  bar_format='{l_bar}{bar:30}{r_bar}', ncols=80, leave=True) as pbar:
            day = d0
            while day <= d1:
                ymd = day.strftime("%Y%m%d")
                for page in range(1, max_pages + 1):
                    # ★ 검증 필요 엔드포인트 ★ (언론사별·날짜별 기사 목록)
                    url = (f"https://news.naver.com/main/list.naver"
                           f"?mode=LPOD&mid=sec&oid={oid}&date={ymd}&page={page}")
                    driver.get(url)
                    time.sleep(1.5)
                    soup = BeautifulSoup(driver.page_source, "html.parser")
                    anchors = soup.select("a[href*='article']")
                    page_links = [a.get("href", "") for a in anchors
                                  if ("n.news.naver.com" in a.get("href", "")
                                      or "/article/" in a.get("href", ""))]
                    page_links = list(set(page_links))
                    if not page_links:
                        break                      # 해당 날짜에 더 이상 기사 없음
                    before = len(links)
                    links.extend(page_links)
                    links = list(set(links))
                    if len(links) == before:
                        break                      # 새 링크가 안 늘면 중단(중복 페이지)
                pbar.update(1)
                pbar.set_postfix({"링크": len(links)})
                day += timedelta(days=1)
    except Exception as e:
        print(f"전체기사 링크 수집 중 오류: {e}")
    finally:
        driver.quit()
    print(f"총 {len(links)}개의 뉴스 링크를 수집했습니다. (oid={oid})")
    return links


def collect_and_save_data(news_offices, query, base_path, start_date, end_date, max_pages=1000, collect_comments=True, collect_image=True, image_quality='medium', separate_image_rows=False, strict_filter=True, comment_workers=5, use_query=True):
    # 출력 디렉토리 생성
    os.makedirs(base_path, exist_ok=True)
    effective_strict = strict_filter and use_query  # 키워드 미사용 시 자동 해제

    # 1. 뉴스 기사 수집
    for media_name, media_id in news_offices.items():
        print(f"\n▶ {media_name} 뉴스 수집 시작...")

        # 언론사별 저장 디렉토리 생성
        media_dir = os.path.join(base_path, media_name)
        os.makedirs(media_dir, exist_ok=True)

        # 기사 데이터 수집
        articles = []
        if use_query:
            links = collect_naver_news_links(query, media_id, start_date, end_date, max_pages)
        else:
            links = collect_office_news_links(media_id, start_date, end_date, max_pages)

        if not links:
            print(f"⚠️ {media_name}에서 수집된 링크가 없습니다.")
            continue

        # tqdm 설정 변경 - 간결한 게이지 바 형식으로 표시 (한 줄로 유지)
        with tqdm(total=len(links), desc=f"{media_name} 뉴스 수집",
                 bar_format='{l_bar}{bar:30}{r_bar}',
                 ncols=80, position=0, leave=True) as pbar:
            for url in links:
                article_details = collect_article_details(media_name, url, collect_image, image_quality)

                # 이미지 행 분리 옵션
                if separate_image_rows and article_details.get('images'):
                    # 이미지가 여러 개면 각 이미지마다 별도 행 생성
                    image_urls = article_details['images'].split('|')
                    if image_urls and image_urls[0]:  # 빈 문자열 체크
                        for img_url in image_urls:
                            # 각 이미지마다 새로운 행 생성
                            article_copy = article_details.copy()
                            article_copy['images'] = img_url  # 하나의 이미지만
                            articles.append(article_copy)
                    else:
                        # 이미지가 없으면 그냥 추가
                        articles.append(article_details)
                else:
                    # 기본 모드: 이미지를 | 로 구분해서 한 행에
                    articles.append(article_details)

                pbar.update(1)

        # 기사 데이터 저장
        article_file = os.path.join(media_dir, "articles.csv")
        df = pd.DataFrame(articles)

        # 빈 데이터 필터링
        valid_articles = df[(df['title'] != "") & (df['text'] != "")]

        # ── 엄격하게 수집 (strict_filter=True 시 검색어 포함 기사만 저장) ──────
        # ★ 버그 수정: title/text 만 검사하면 놓친다.
        #   본문 분해 단계에서 부제(strong.media_end_summary 등)와 기사 서두의
        #   굵은 글씨 리드문("가짜 부제")을 text에서 걷어내는데, 검색어가 하필
        #   그 리드문/부제 안에만 있는 기사가 많으면(예: "5·18" 관련 기사는
        #   서두에 굵게 강조된 리드 문장에 검색어가 오는 경우가 흔하다)
        #   text 에서는 검색어가 사라져 title 도 못 맞고 text 도 못 맞아
        #   전부 0건으로 걸러지는 문제가 생긴다.
        #   → subtitle/captions/subheads 까지 함께 검사해 걷어낸 텍스트에
        #     있던 검색어도 놓치지 않는다. (2번째 코드의 apply_strict_filter 가
        #     title+subtitle+fulltext 를 하나로 합쳐 검사하는 것과 같은 이유)
        if effective_strict:
            keyword = query.strip('"')  # 검색어에서 따옴표 제거 후 필터 키워드로 사용
            before_count = len(valid_articles)
            hay_cols = [c for c in ("title", "subtitle", "captions", "subheads", "text")
                        if c in valid_articles.columns]
            mask = False
            for col in hay_cols:
                mask = mask | valid_articles[col].str.contains(keyword, na=False, regex=False)
            valid_articles = valid_articles[mask]
            print(f"  [엄격하게 수집] '{keyword}' 포함 기사({'+'.join(hay_cols)} 기준): "
                  f"{before_count}개 → {len(valid_articles)}개")

        # 저장
        import csv as _csv
        valid_articles.to_csv(article_file, index=False, encoding="utf-8-sig", quoting=_csv.QUOTE_ALL)
        print(f"✓ {len(valid_articles)}개의 기사를 {article_file}에 저장했습니다.")

    print("\n★ 모든 뉴스 수집이 완료되었습니다! ★")

    # 2. 댓글 수집 (설정이 활성화된 경우) — 전수 수집 로직 사용
    if collect_comments:
        safe_query = sanitize_filename(query)
        print("\n▶ 댓글 전수 수집을 시작합니다...")
        collect_and_save_comments(base_path, safe_query, max_workers=comment_workers)

def sanitize_filename(filename):
    """
    파일 이름에 사용할 수 없는 문자를 제거하거나 대체하는 함수
    """
    # 윈도우에서 파일 이름으로 사용할 수 없는 문자
    invalid_chars = ['\\', '/', ':', '*', '?', '"', '<', '>', '|']
    for ch in invalid_chars:
        filename = filename.replace(ch, '_')
    return filename.strip() or "전체기사"



# ══════════════════════════════════════════════════════════════════════════════
# 다중 키워드 수집 (키워드별 서브폴더)
# ══════════════════════════════════════════════════════════════════════════════

def _run_single_keyword(args):
    (kw, news_offices, base_path, start_date, end_date,
     max_pages, collect_comments, collect_image,
     image_quality, separate_image_rows, strict_filter, comment_workers, use_query) = args

    kw_clean = kw.strip('"').strip()
    kw_dir   = os.path.join(base_path, kw_clean)
    os.makedirs(kw_dir, exist_ok=True)

    print(f"\n{'='*70}")
    print(f"  키워드: [{kw_clean}]  폴더: {kw_dir}")
    print(f"{'='*70}")

    collect_and_save_data(
        news_offices, kw, kw_dir,
        start_date, end_date, max_pages,
        collect_comments, collect_image,
        image_quality, separate_image_rows,
        strict_filter, comment_workers, use_query
    )


def collect_multi_keywords(
    queries,
    news_offices,
    base_path,
    start_date,
    end_date,
    max_pages           = 10000,
    collect_comments    = True,
    collect_image        = False,
    image_quality        = "low",
    separate_image_rows  = False,
    strict_filter        = True,
    keyword_workers      = 1,
    comment_workers       = 5,
    use_query            = True,
):
    os.makedirs(base_path, exist_ok=True)
    if not use_query:
        queries = ["전체기사"]   # 키워드 미사용: 언론사 전체 기사(폴더명 placeholder)
    print(f"수집 키워드: {queries}")
    print(f"처리 방식: {'병렬 ' + str(keyword_workers) + '개' if keyword_workers > 1 else '순차'}")

    args_list = [
        (kw, news_offices, base_path, start_date, end_date,
         max_pages, collect_comments, collect_image,
         image_quality, separate_image_rows, strict_filter, comment_workers, use_query)
        for kw in queries
    ]

    if keyword_workers == 1:
        for args in args_list:
            _run_single_keyword(args)
    else:
        workers = min(keyword_workers, len(queries))
        with ThreadPoolExecutor(max_workers=workers) as ex:
            futs = [ex.submit(_run_single_keyword, a) for a in args_list]
            for fut in _as_completed(futs):
                try:
                    fut.result()
                except Exception as e:
                    print(f"키워드 오류: {e}")

    print(f"\n★ 전체 키워드 수집 완료 ★")


# ══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    # ============================================================
    # 네이버 제휴 언론사 84개 고유번호 목록
    # 고유번호 확인 방법:
    #   1. 해당 언론사의 네이버뉴스 기사 URL 확인
    #   2. https://n.news.naver.com/mnews/article/001/0015350225 에서 001이 언론사 ID
    #   3. 고유번호 = 1 + 언론사 ID (예: 001 → 1001)
    # 사용 방법: 수집할 언론사 앞의 # 을 제거하세요.
    # ============================================================
    news_offices = {
    # ── 방송사 ──────────────────────────────────────────────────────────────
    # ✓ = GitHub/Naver URL 직접 확인 | ⚠️ = 원본값 유지, 실사용 전 URL 직접 검증 권장
    "KBS":          1056,   # ✓ oid=056
    "MBC":          1214,   # ✓ oid=214
    "SBS":          1055,   # ✓ oid=055
    "YTN":          1052,   # ✓ oid=052
    "연합뉴스TV":   1422,   # ✓oid=422 (원본 유지)
    "MBN":          1019,   # ✓ oid=019
    "TV조선":       1448,   # ✓ oid=448
    "채널A":        1449,   # ✓ oid=449 (원본 유지)
    "JTBC":         1437,   # ✓ oid=437

    # ── 종합일간지 ──────────────────────────────────────────────────────────
    "경향신문":     1032,   # ✓ oid=032
    "국민일보":     1005,   # ✓ oid=005
    "동아일보":     1020,   # ✓ oid=020
    #"문화일보":     1021,   # ✓ oid=021
    "서울신문":     1081,   # ✓ oid=081
    #"세계일보":     1022,   # ✓ oid=022
    "조선일보":     1023,   # ✓ oid=023
    "중앙일보":     1025,   # ✓ oid=025
    "한겨레":       1028,   # ✓ oid=028
    "한국일보":     1469,   # ✓ oid=469

    # ── 경제지 ──────────────────────────────────────────────────────────────
    "매일경제":     1009,   # ✓ oid=009
    "머니투데이":   1008,   # ✓ oid=008
    "서울경제":     1011,   # ✓ oid=011
    "아시아경제":   1277,   # ✓ oid=277
    "이데일리":     1018,   # ✓ oid=018
    "파이낸셜뉴스": 1014,   # ✓ oid=014
    "한국경제":     1015,   # ✓ oid=015
    #"한국경제tv": 1004,   # ✓ oid=004
    "헤럴드경제":   1016,   # ✓ oid=016
    "조선비즈":     1366,   # ✓ oid=366
    #"한경비즈니스": 1050,   # ✓ oid=050,
    #"SBS Biz":      1374,   # ✓ oid=374,
    #"비즈워치": 1648,       # ✓ oid=648 (648=비즈워치)
    #"매경이코노미": 1024,    # ✓ oid=024
    #"이코노미스트": 1243,   # ✓ oid=243
    #"코리아헤럴드": 1044,

    # ── 통신사 / 인터넷 매체 ────────────────────────────────────────────────
    "연합뉴스":     1001,   # ✓ oid=001
    #"뉴시스":       1003,   # ✓ oid=003
    #"뉴스1":        1421,   # ✓ oid=421
    #"오마이뉴스":   1047,   # ✓ oid=047
    # "프레시안":     1002,   # ✓ oid=002 (원본 1143은 오류)
    #"미디어오늘":   1006,   # ✓ oid=006
    #"노컷뉴스":     1079,   # ✓ oid=079 (CBS와 동일)
    #"데일리안":     1119,   # ✓ oid=119
    #"아이뉴스24":   1031,   # ✓ oid=031 (원본 1109는 OSEN으로 오류)
    #"뉴스타파": 1607,     # ✓ oid=607 (607=뉴스타파),
    #"더팩트": 1629, # ✓ oid=629 (629=더팩트)

    # ── 주간지 / 잡지 ───────────────────────────────────────────────────────
    #"한겨레21":     1036,   # ✓ oid=036 (원본 1227은 아시아경제로 오류)
    #"주간경향":     1033,   # ✓ oid=033 (원본 1307은 오류)
    #"주간동아":     1037,   # ✓ oid=037 (원본 1020은 동아일보로 오류)
    #"주간조선":     1053,   # ✓ oid=053 (원본 유지)
    #"시사저널":     1586,   # ✓ oid=586 (원본 1308은 오류)
    #"시사IN":       1308,   # ✓ oid=308 (원본 유지)
    #"신동아":       1262,   # ✓ oid=262, 별도 oid 없을 수 있음

    # ── 지역일간지 ──────────────────────────────────────────────────────────
    # 아래 지역지는 실사용 전 반드시 해당 언론사 네이버뉴스 URL의 oid 직접 확인 권장
    #"부산일보":     1082,   # ✓ oid=082
    #"국제신문":     1658,   # ✓ oid=658 (원본 유지)
    #"경기일보":     1666,   # ✓ oid=666 (원본 유지)
    #"강원도민일보": 1654,   # ✓ oid=654 (원본 유지)
    #"강원일보":     1087,   # ✓ oid=087 (원본 유지)
    #"대전일보":     1656,   # ✓ oid=090 (원본 유지)
    #"매일신문":     1088,   # ✓ oid=088 (원본 1093은 오류)
    #"제주일보":     1084,   # ✓ oid=084 (원본 1107은 오류 → 084=제주일보사)
    #"kbc광주방송": 1660,   # ✓ oid=660 (660=kbc광주방송)
    #"CJB청주방송": 1655, # ✓ oid=655 (655=CJB청주방송),
    #"전주MBC": 1659, # ✓ oid=659 (659=전주MBC)
    #"대구MBC": 1657, # ✓ oid=657 (657=대구MBC)


    # ── IT / 전문지 ─────────────────────────────────────────────────────────
    #"지디넷코리아": 1092,   # ✓ ZDNet Korea와 동일 매체 (중복 수집 주의)
    #"전자신문":     1030,   # ✓ oid=030
    #"디지털타임스": 1029,   # ✓ oid=029
    #"블로터":       1293,   # ✓ oid=293
    #"디지털데일리": 1138,   # ✓ oid=138 (원본 유지)
    #"농민신문": 1662,# ✓ oid=662
    #"코메디닷컴": 1296,
    #"헬스조선": 1346,

    # ── 스포츠 / 연예 ────────────────────────────────────────────────────────
    #"스포츠조선":   1076,   # ✓ oid=076
    #"스포츠동아":   1382,   # ✓ oid=139 (원본 유지)
    #"스포츠서울":   1468,   # ✓ 원본 1076은 스포츠조선으로 오류 → URL 직접 확인 필요
    #"일간스포츠":   1241,   # ✓ oid=241 (원본 유지)
    #"OSEN":         1109,   # ✓ oid=109
    #"스타뉴스":     1108,   # ✓ oid=108
}

    # ============================================================
    # 수집 설정  ★ 이 블록만 수정하세요 ★
    # ============================================================

    # ── 검색 키워드 ─────────────────────────────────────────────
    # 단일:  queries = ["5·18"]
    # 다중:  queries = ["중국", "아프리카"]
    queries = ["5.18", "5·18"]

    base_path   = r"C:\Users\com\Desktop\5185월1"  # 최상위 저장 경로
    start_date  = "20260501"                             # 수집 시작일 (YYYYMMDD)
    end_date    = "20260531"                             # 수집 종료일 (YYYYMMDD)
    max_pages   = 10000

    # ── 키워드 병렬 옵션 ─────────────────────────────────────────
    keyword_workers = 2   # 1: 순차(권장) / 2~: Chrome N개 동시

    # ── 이미지 수집 옵션 ─────────────────────────────────────────
    collect_image       = False  # True: 이미지 수집 / False: 미수집
    image_quality       = "low"  # "high"(원본) / "medium"(w860) / "low"(w647)
    separate_image_rows = False  # True: 이미지별 행 분리 / False: | 구분 한 행

    # ── 댓글 수집 옵션 ───────────────────────────────────────────
    collect_comments = True     # True: 댓글 수집(전수) / False: 미수집
    comment_workers  = 5        # 댓글 병렬 스레드 수 (기본 5, 차단 우려 시 3)

    # ── 엄격하게 수집 옵션 ──────────────────────────────────────────
    use_query = True            # ★ True: 검색어 포함 기사만 / False: 언론사 전체 기사 수집
    strict_filter = True        # True: 제목·본문에 검색어 포함 기사만 저장 (엄격하게 수집)
                                # False: 네이버 검색 결과 전체 저장

    # ── 실행 ────────────────────────────────────────────────────
    collect_multi_keywords(
        queries             = queries,
        news_offices        = news_offices,
        base_path           = base_path,
        start_date          = start_date,
        end_date            = end_date,
        max_pages           = max_pages,
        collect_comments    = collect_comments,
        collect_image       = collect_image,
        image_quality       = image_quality,
        separate_image_rows = separate_image_rows,
        strict_filter       = strict_filter,
        keyword_workers     = keyword_workers,
        comment_workers     = comment_workers,
        use_query           = use_query,
    )
